# 02 - Function tools and business calculations

A model chooses a typed Python function, ADK executes it, and the model explains the result. Keep monetary calculations in Python.

**Runtime:** ADK 2.10.0, your existing `dataengenv` kernel and Application Default Credentials (ADC).

**Cost:** Gemini calls, including tool follow-up responses. All data below is synthetic. Run top to bottom; no `course.py`, configuration file or data file is needed.

## Initialize

Set your project, location and model below. As in notebook 01, the client uses standard TLS certificate verification.

In [ ]:
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT = "gen-lang-client-0367558705"
LOCATION = "global"
MODEL = "gemini-2.5-flash-lite"
APP_NAME = "ecommerce_lab"
USER_ID = "customer_101"

model = Gemini(
    model=MODEL,
    client_kwargs={"vertexai": True, "project": PROJECT, "location": LOCATION},
)

## Define the data and tools

Type hints describe inputs; docstrings tell the model what each tool does. Unknown IDs return a structured error. The 18% GST rate is only a demo assumption.

In [ ]:
from decimal import Decimal

ORDERS = [{'order_id': 'O1001',
  'customer_id': 'C101',
  'status': 'delivered',
  'days_since_delivery': 5,
  'items': [{'product_id': 'P101', 'quantity': 2, 'unit_price': 1299},
            {'product_id': 'P102', 'quantity': 1, 'unit_price': 899}],
  'subtotal': 3497,
  'tax': 629.46,
  'total': 4126.46,
  'currency': 'INR',
  'invoice_id': 'INV1001'},
 {'order_id': 'O1002',
  'customer_id': 'C102',
  'status': 'processing',
  'days_since_delivery': None,
  'items': [{'product_id': 'P102', 'quantity': 1, 'unit_price': 899}],
  'subtotal': 899,
  'tax': 161.82,
  'total': 1060.82,
  'currency': 'INR',
  'invoice_id': 'INV1002'}]

PRODUCTS = [{'product_id': 'P101', 'name': 'USB-C Hub', 'price': 1299, 'currency': 'INR', 'stock': 12},
 {'product_id': 'P102', 'name': 'Laptop Stand', 'price': 899, 'currency': 'INR', 'stock': 5},
 {'product_id': 'P103', 'name': 'Wireless Mouse', 'price': 599, 'currency': 'INR', 'stock': 0}]

In [ ]:
def get_order(order_id: str) -> dict:
    """Look up a synthetic order by ID, including line items and invoice total."""
    return next((dict(order) for order in ORDERS if order["order_id"] == order_id),
                {"error": "Order not found"})


def get_product(product_id: str) -> dict:
    """Look up a synthetic product's name, price and stock by ID."""
    return next((dict(product) for product in PRODUCTS if product["product_id"] == product_id),
                {"error": "Product not found"})


def calculate_order_total(order_id: str) -> dict:
    """Calculate an order's subtotal, demo GST and total from its line items."""
    order = get_order(order_id)
    if "error" in order:
        return order
    subtotal = sum(
        (Decimal(str(item["unit_price"])) * item["quantity"] for item in order["items"]),
        Decimal("0.00"),
    )
    tax = (subtotal * Decimal("0.18")).quantize(Decimal("0.01"))
    return {"order_id": order_id, "subtotal": str(subtotal), "tax": str(tax),
            "total": str(subtotal + tax), "currency": "INR"}

## Check the tools locally

These calls use only Python and do not contact the model.

In [ ]:
print(calculate_order_total("O1001"))
print(calculate_order_total("O1002"))
print(get_order("UNKNOWN"))

## Give the tools to the agent

The Runner handles function calls and sends their results back to the model. Print tool calls and the final response to see that flow.

In [ ]:
agent = Agent(
    name="order_tools",
    model=model,
    instruction="Always use tools for order facts and arithmetic. Report currency and total.",
    tools=[get_order, get_product, calculate_order_total],
    generate_content_config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=768,
    ),
)

session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

message = types.Content(
    role="user",
    parts=[types.Part(text="Look up O1001 and calculate its total including GST.")],
)
events = []
async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message,
):
    events.append(event)
    if event.error_code:
        raise RuntimeError(f"{event.error_code}: {event.error_message}")
    for call in event.get_function_calls():
        print(f"Tool: {call.name}({call.args})")
    if event.is_final_response() and event.content:
        answer = "".join(part.text or "" for part in event.content.parts or [])
        print("Final answer:", answer)

## Try it

Change the prompt to ask for O1002, then an unknown order, and rerun the agent cell. Inspect tool calls and structured error returns.

## Reference

[Official ADK function tools documentation](https://adk.dev/tools-custom/function-tools/). Configuration and sample data are above. Use your existing authenticated kernel; no activation or login cells are needed.